# Day 22 – Regression Diagnostics & Model Evaluation
## From “We Built a Model” to “Can We Trust It?”

Part of the **30 Days of Data Analytics** series.

**Framework:** Build → Diagnose → Evaluate → Validate → Decide

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.random.seed(42)

## 2. Create a Synthetic Public Grievance Dataset

This synthetic dataset represents monthly complaint volume and average resolution time.

In [ ]:
months = pd.date_range("2024-01-01", periods=36, freq="MS")

complaints = np.array([
    420,450,470,510,530,560,590,610,640,680,700,730,
    760,790,820,850,890,910,940,970,1000,1040,1080,1120,
    1150,1180,1210,1250,1290,1320,1360,1390,1430,1470,1510,1560
])

resolution_days = (
    2.5 + 0.0055 * complaints +
    np.random.normal(0, 0.7, len(complaints))
)

df = pd.DataFrame({
    "month": months,
    "complaints": complaints,
    "resolution_days": resolution_days
})

df.head()

## 3. Explore the Relationship

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["complaints"], df["resolution_days"])
plt.xlabel("Monthly Complaint Volume")
plt.ylabel("Average Resolution Time (Days)")
plt.title("Complaint Volume vs Resolution Time")
plt.grid(True, alpha=0.3)
plt.show()

## 4. Build a Regression Model

In [ ]:
X = df[["complaints"]]
y = df["resolution_days"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"Intercept: {model.intercept_:.3f}")
print(f"Slope: {model.coef_[0]:.5f}")

## 5. Evaluate Training and Testing Performance

Key metrics:
- **R²** – proportion of variation explained
- **MAE** – average absolute prediction error
- **RMSE** – penalizes larger errors more strongly

In [ ]:
results = pd.DataFrame({
    "Metric": ["R²", "MAE", "RMSE"],
    "Training": [
        r2_score(y_train, y_train_pred),
        mean_absolute_error(y_train, y_train_pred),
        np.sqrt(mean_squared_error(y_train, y_train_pred))
    ],
    "Testing": [
        r2_score(y_test, y_test_pred),
        mean_absolute_error(y_test, y_test_pred),
        np.sqrt(mean_squared_error(y_test, y_test_pred))
    ]
})

results.round(3)

## 6. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test, y_test_pred)

line_min = min(y_test.min(), y_test_pred.min())
line_max = max(y_test.max(), y_test_pred.max())

plt.plot(
    [line_min, line_max],
    [line_min, line_max],
    linestyle="--"
)

plt.xlabel("Actual Resolution Time (Days)")
plt.ylabel("Predicted Resolution Time (Days)")
plt.title("Actual vs Predicted")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Residual Analysis

**Residual = Actual − Predicted**

A good residual plot should generally show observations scattered around zero without an obvious systematic pattern.

In [ ]:
test_results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_test_pred
})

test_results["residual"] = (
    test_results["actual"] - test_results["predicted"]
)

plt.figure(figsize=(8,5))
plt.scatter(test_results["predicted"], test_results["residual"])
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Cross-Validation

Cross-validation gives a more robust estimate of model performance by repeatedly training and evaluating the model on different data partitions.

In [ ]:
cv_scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² scores:", np.round(cv_scores, 3))
print(f"Mean R²: {cv_scores.mean():.3f}")
print(f"Standard deviation: {cv_scores.std():.3f}")

## 9. Overfitting and Underfitting

**Overfitting:** training performance is very strong, but performance on unseen data is substantially worse.

**Underfitting:** the model performs poorly on both training and testing data.

The objective is good **generalization** to unseen data.

## 10. Regression Diagnostics

Important checks include:

1. Linearity
2. Independence
3. Homoscedasticity
4. Residual distribution
5. Outliers and influential observations
6. Multicollinearity in multiple regression

A high R² alone does not guarantee a reliable model.

## 11. Outlier Investigation

Do not automatically remove unusual observations. First determine whether an outlier is:

- a data-entry error
- a genuine exceptional event
- a service disruption
- an important operational case

In public-sector data, unusual observations can contain valuable information.

In [ ]:
# Flag the largest absolute test residuals
test_results["absolute_residual"] = test_results["residual"].abs()

test_results.sort_values(
    "absolute_residual",
    ascending=False
).head()

## 12. Public Governance Applications

Regression diagnostics and model evaluation can support:

- Complaint resolution-time prediction
- Workload planning
- Resource allocation
- Service-level monitoring
- Ward and zone performance analysis
- Early-warning systems

The objective is not simply to maximize R².

The objective is:

**Reliable prediction → Better operational decision → Better citizen service**

## 13. Model Evaluation Checklist

**Build → Diagnose → Evaluate → Validate → Decide**

Before operational use:

- Evaluate on unseen data
- Compare MAE, RMSE and R²
- Inspect residuals
- Investigate outliers
- Check assumptions
- Use cross-validation where appropriate
- Compare alternative models
- Confirm operational usefulness
- Monitor performance after deployment

## 14. Key Takeaway

> **A model is not good because it produces predictions. A model is good when those predictions are reliable, validated, interpretable and useful for decisions.**

### Analytics Journey

**Data → Relationship → Model → Prediction → Validation → Decision**

### Next
**Day 23 – Feature Engineering: Turning Raw Data into Useful Predictors**